# 04 - 语雀更新与删除细粒度测试

> 测试目标：验证更新操作的各种组合方式，以及边界情况处理。
> 测试内容：只更新标题、只更新内容、同时更新、空内容、长文档、特殊字符。

## 测试目录

1. [环境准备](#环境准备)
2. [只更新标题](#只更新标题)
3. [只更新内容](#只更新内容)
4. [同时更新标题和内容](#同时更新标题和内容)
5. [边界测试：空内容](#边界测试空内容)
6. [边界测试：长文档](#边界测试长文档)
7. [边界测试：特殊字符](#边界测试特殊字符)
8. [清理](#清理)

---

## 环境准备

In [ ]:
# =============================================================================
# 环境准备
# =============================================================================

import os
from pathlib import Path
from datetime import datetime

try:
    from dotenv import load_dotenv
    load_dotenv(Path('../../.env'))
except ImportError:
    pass

from yuque_client import YuqueClient, lake_html

client = YuqueClient()
print('✅ 客户端初始化成功')

books = client.list_books()
BOOK_ID = books[0]['id']
print(f'📚 使用知识库: {books[0]["name"]} (ID={BOOK_ID})')

# 记录所有测试文档
TEST_DOC_IDS = []

---

## 创建基础测试文档

先创建一个文档，用于后续的更新测试。

In [ ]:
# =============================================================================
# 创建基础文档
# =============================================================================

base_doc = client.create_doc(
    book_id=BOOK_ID,
    title=f'[更新测试]基础文档-{datetime.now().strftime("%H%M%S")}',
    content=lake_html(
        '<h1>原始标题</h1>',
        '<p>原始内容。</p>',
        '<ul><li>原始列表项</li></ul>',
    ),
)

BASE_DOC_ID = base_doc['id']
BASE_DOC_SLUG = base_doc['slug']
TEST_DOC_IDS.append(BASE_DOC_ID)

print(f'✅ 基础文档创建成功')
print(f'   ID: {BASE_DOC_ID}')
print(f'   Slug: {BASE_DOC_SLUG}')

---

## 只更新标题

PUT 请求只传 `title`，不传 `body_asl`，验证标题更新而内容保持不变。

In [ ]:
# =============================================================================
# 只更新标题
# =============================================================================

new_title = f'[更新测试]只改标题-{datetime.now().strftime("%H%M%S")}'

updated = client.update_doc(
    doc_id=BASE_DOC_ID,
    title=new_title,
)

# 读取验证
doc = client.read_doc(BOOK_ID, BASE_DOC_SLUG)
content = doc.get('content', '')

print(f'只更新标题:')
print(f'   新标题: {doc.get("title")}')
print(f'   标题正确: {"✅" if doc.get("title") == new_title else "❌"}')
print(f'   内容仍包含"原始内容": {"✅" if "原始内容" in content else "❌"}')
print(f'   内容仍包含"原始列表项": {"✅" if "原始列表项" in content else "❌"}')

---

## 只更新内容

PUT 请求只传 `body_asl`，不传 `title`，验证内容更新而标题保持不变。

In [ ]:
# =============================================================================
# 只更新内容
# =============================================================================

new_content = lake_html(
    '<h1>原始标题</h1>',
    '<p>这是更新后的内容，添加了<strong>加粗文字</strong>。</p>',
)

updated = client.update_doc(
    doc_id=BASE_DOC_ID,
    content=new_content,
)

# 读取验证
doc = client.read_doc(BOOK_ID, BASE_DOC_SLUG)
content = doc.get('content', '')

print(f'只更新内容:')
print(f'   标题未变: {"✅" if doc.get("title") == new_title else "❌"}')
print(f'   包含"更新后的内容": {"✅" if "更新后的内容" in content else "❌"}')
print(f'   包含"加粗文字": {"✅" if "加粗文字" in content else "❌"}')
print(f'   旧内容已移除: {"✅" if "原始列表项" not in content else "❌"}')

---

## 同时更新标题和内容

PUT 请求同时传 `title` 和 `body_asl`，验证两者同时更新。

In [ ]:
# =============================================================================
# 同时更新标题和内容
# =============================================================================

final_title = f'[更新测试]最终版-{datetime.now().strftime("%H%M%S")}'
final_content = lake_html(
    '<h1>最终标题</h1>',
    '<p>最终内容。</p>',
    '<ol><li>最终有序项</li></ol>',
)

updated = client.update_doc(
    doc_id=BASE_DOC_ID,
    title=final_title,
    content=final_content,
)

# 读取验证
doc = client.read_doc(BOOK_ID, BASE_DOC_SLUG)
content = doc.get('content', '')

print(f'同时更新标题和内容:')
print(f'   标题正确: {"✅" if doc.get("title") == final_title else "❌"}')
print(f'   包含"最终标题": {"✅" if "最终标题" in content else "❌"}')
print(f'   包含"最终有序项": {"✅" if "最终有序项" in content else "❌"}')
print(f'   旧内容已移除: {"✅" if "加粗文字" not in content else "❌"}')

---

## 边界测试：空内容

创建文档时不传 `content`，只传 `title`，验证是否成功。

In [ ]:
# =============================================================================
# 边界测试：空内容（不传 content）
# =============================================================================

empty_doc = client.create_doc(
    book_id=BOOK_ID,
    title=f'[边界测试]空内容-{datetime.now().strftime("%H%M%S")}',
    # 不传 content
)

if empty_doc:
    TEST_DOC_IDS.append(empty_doc['id'])
    doc = client.read_doc(BOOK_ID, empty_doc['slug'])
    content_len = len(doc.get('content', ''))
    print(f'✅ 空内容文档创建成功')
    print(f'   ID: {empty_doc["id"]}')
    print(f'   content 长度: {content_len} chars')
else:
    print(f'❌ 空内容文档创建失败')

---

## 边界测试：长文档

创建包含约 5000 字的长文档，验证 API 对大文档的处理能力。

In [ ]:
# =============================================================================
# 边界测试：长文档（~5000 字）
# =============================================================================

paragraphs = []
for i in range(50):
    paragraphs.append(f'<p>这是第{i+1}段长文本。用于测试语雀 API 对大文档的处理能力。'*5 + '</p>')

long_content = lake_html('<h1>长文档测试</h1>', *paragraphs)

long_doc = client.create_doc(
    book_id=BOOK_ID,
    title=f'[边界测试]长文档-{datetime.now().strftime("%H%M%S")}',
    content=long_content,
)

if long_doc:
    TEST_DOC_IDS.append(long_doc['id'])
    print(f'✅ 长文档创建成功')
    print(f'   ID: {long_doc["id"]}')
    print(f'   提交内容长度: {len(long_content)} 字符')
    
    # 读取验证完整性
    doc = client.read_doc(BOOK_ID, long_doc['slug'])
    content = doc.get('content', '')
    print(f'   读取内容长度: {len(content)} 字符')
    
    has_start = '第1段' in content
    has_end = '第50段' in content
    print(f'   包含开头（第1段）: {"✅" if has_start else "❌"}')
    print(f'   包含结尾（第50段）: {"✅" if has_end else "❌"}')
    
    if has_start and has_end:
        print(f'   ✅ 内容完整性验证通过')
    else:
        print(f'   ❌ 内容不完整！')
else:
    print(f'❌ 长文档创建失败')

---

## 边界测试：特殊字符

测试 HTML 实体、中文、emoji、数学符号等特殊字符。

In [ ]:
# =============================================================================
# 边界测试：特殊字符
# =============================================================================

special_content = lake_html(
    '<h1>特殊字符测试</h1>',
    '<p>HTML实体: &lt;div&gt; &amp; &quot;quote&quot;</p>',
    '<p>中文：这是一段中文文本，包含简体中文和繁體中文。</p>',
    '<p>Emoji: 🎉 🚀 💡 ✅ ❌</p>',
    '<p>数学符号: α β γ δ ε ζ η θ</p>',
)

special_doc = client.create_doc(
    book_id=BOOK_ID,
    title=f'[边界测试]特殊字符-{datetime.now().strftime("%H%M%S")}',
    content=special_content,
)

if special_doc:
    TEST_DOC_IDS.append(special_doc['id'])
    doc = client.read_doc(BOOK_ID, special_doc['slug'])
    content = doc.get('content', '')
    
    print(f'✅ 特殊字符文档创建成功')
    print(f'   ID: {special_doc["id"]}')
    
    checks = [
        ('特殊字符测试' in content, '包含中文标题'),
        ('中文' in content, '包含中文正文'),
        ('α β γ' in content, '包含数学符号'),
    ]
    for passed, desc in checks:
        print(f'   {"✅" if passed else "❌"} {desc}')
else:
    print(f'❌ 特殊字符文档创建失败')

---

## 清理

删除所有测试文档。

In [ ]:
# =============================================================================
# 清理所有测试文档
# =============================================================================

print(f'准备清理 {len(TEST_DOC_IDS)} 个测试文档...\n')

success = 0
fail = 0
for doc_id in TEST_DOC_IDS:
    if client.delete_doc(doc_id, BOOK_ID):
        print(f'✅ 已删除: {doc_id}')
        success += 1
    else:
        print(f'❌ 删除失败: {doc_id}')
        fail += 1

print(f'\n清理完成: {success} 成功, {fail} 失败')

---

## 附录：PUT 更新行为总结

| 更新方式 | 传 title | 传 body_asl | 结果 |
|----------|---------|------------|------|
| 只更新标题 | ✅ | ❌ | 标题变更，内容不变 |
| 只更新内容 | ❌ | ✅ | 内容变更，标题不变 |
| 同时更新 | ✅ | ✅ | 标题和内容都变更 |

**注意**：
- 不传某个字段时，该字段保持原值
- body_asl 会完全替换原有内容，不是追加
- 更新后读取的 slug 可能变化（如果标题变了），但 doc_id 不变